# 人机交互
人机交互中间件 (HITL) 允许您在代理工具调用中添加人工监督。当模型提出可能需要审核的操作（例如，写入文件或执行 SQL）时，中间件可以暂停执行并等待决策。
它通过检查每个工具调用是否符合可配置的策略来实现这一点。如果需要干预，中间件会发出中断以暂停执行。图状态使用 LangGraph 的持久层保存，因此执行可以安全地暂停并在稍后恢复。

然后，由人来决定接下来会发生什么：该操作可以按原样批准（`approve`），在运行之前进行修改（`edit`），或者被拒绝并给予反馈（`reject`）。


## 中断决策类型
中间件定义了人类响应中断的三种内置方式：

| 决策类型 | 描述 | 示例用例 |
| :--- | :--- | :--- |
| approve | 该行动按原样批准并执行，未作任何更改。 | 请按原样发送电子邮件草稿。 |
| edit | 工具调用经过修改后执行。 | 发送电子邮件前请更改收件人 |
| Xreject | 工具调用被拒绝，并在对话中添加了解释。 | 驳回邮件草稿并解释如何修改。 |

每个工具的可用决策类型取决于您在interrupt_on 中配置的策略。 当多个工具调用同时暂停时，每个操作都需要单独的决策。 决策必须按照中断请求中出现的操作的顺序提供。

> 编辑工具参数时，请谨慎修改。对原始参数进行重大修改可能会导致模型重新评估其方法，并可能多次执行该工具或采取意外操作。

## 配置中断
要使用 HITL，请在创建代理时将中间件添加到代理的middleware列表中。

您可以通过将工具操作映射到每个操作允许的决策类型来进行配置。当工具调用与映射中的某个操作匹配时，中间件将中断执行。


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="gpt-4o",
    tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject) allowed
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                # Safe operation, no approval needed
                "read_data": False,
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),  
)

您必须配置检查指针以跨中断保持图形状态。 在生产中，使用持久检查指针，例如 AsyncPostgresSaver。 对于测试或原型设计，请使用 InMemorySaver。

## 应对中断
调用代理后，它会一直运行，直到完成或触发中断。当工具调用与您配置的策略匹配时，就会触发中断。interrupt_on在这种情况下，调用结果将包含一个__interrupt__字段，其中列出了需要审核的操作。然后，您可以将这些操作提交给审核人员，并在获得决策后恢复执行。

In [ ]:
from langgraph.types import Command

# Human-in-the-loop leverages LangGraph's persistence layer.
# You must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
config = {"configurable": {"thread_id": "some_id"}} 
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete old records from the database",
            }
        ]
    },
    config=config 
)

# The interrupt contains the full HITL request with action_requests and review_configs
print(result['__interrupt__'])
# > [
# >    Interrupt(
# >       value={
# >          'action_requests': [
# >             {
# >                'name': 'execute_sql',
# >                'arguments': {'query': 'DELETE FROM records WHERE created_at < NOW() - INTERVAL \'30 days\';'},
# >                'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {...}'
# >             }
# >          ],
# >          'review_configs': [
# >             {
# >                'action_name': 'execute_sql',
# >                'allowed_decisions': ['approve', 'reject']
# >             }
# >          ]
# >       }
# >    )
# > ]


# Resume with approval decision
agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "edit", "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

## 决策类型


In [ ]:
## 用于`approve`批准工具调用并按原样执行，无需更改。
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "approve",
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

In [ ]:
### 用于edit在执行前修改工具调用。提供修改后的操作、新的工具名称和参数。

agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "new_tool_name",
                        # Arguments to pass to the tool.
                        "args": {"key1": "new_value", "key2": "original_value"},
                    }
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

In [ ]:
# 用于reject拒绝工具调用并提供反馈而不是执行。
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # listed in the `__interrupt__` request.
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation about why the action was rejected
                    "message": "No, this is wrong because ..., instead do this ...",
                }
            ]
        }
    ),
    config=config  # Same thread ID to resume the paused conversation
)

这条信息message会作为反馈添加到对话中，帮助代理理解为什么操作被拒绝以及应该采取什么替代措施。

### 多项决定
当需要审查多个操作时，请按照中断中出现的顺序，对每个操作做出决定：

```json
{
    "decisions": [
        {"type": "approve"},
        {
            "type": "edit",
            "edited_action": {
                "name": "tool_name",
                "args": {"param": "new_value"}
            }
        },
        {
            "type": "reject",
            "message": "This action is not allowed"
        }
    ]
}

```

## 执行生命周期
中间件定义了一个after_model钩子，该钩子在模型生成响应之后、任何工具调用执行之前运行：
- 代理调用模型生成响应。
- 中间件会检查响应中是否存在工具调用。
- 如果任何调用需要人工输入，中间件会构建一个HITLRequest带有action_requests和的，review_configs并调用中断。
- 智能体等待人类做出决定。
- 根据这些HITLResponse决定，中间件执行已批准或已编辑的调用，合成被拒绝调用的ToolMessage ，并恢复执行。
​
## 自定义 HITL 逻辑
- 对于更专业的流程，您可以直接使用中断原语和中间件抽象来构建自定义 HITL 逻辑。
- 查看上面的执行生命周期，了解如何将中断集成到代理的操作中。